# Ticket Data Analysis

Loads and explores the ~20k work item (Incident/Request/etc.) records from a JSON file.

Set `DATA_PATH` below to point at your JSON file.

The second half of the notebook analyses the **Assignee** field: is there any pattern in who gets which ticket, and can a model predict it?

In [2]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

In [4]:
# Path to the JSON file with the work item records (adjust as needed)
DATA_PATH = Path("jira_first_20000_requested_fields_synthetic.json")

with open(DATA_PATH, "r", encoding="utf-8") as f:
    records = json.load(f)

print(f"Loaded {len(records):,} records")
records[0]

Loaded 20,000 records


{'Work type': 'Service Request',
 'Summary': 'New license requested for Tax Reporting',
 'Description': 'A new user requires a license for Tax Reporting to support an operational activity. The request includes the relevant user details and justification and needs validation before the entitlement is assigned.',
 'Affected Business or IT Services': ['Tax Reporting'],
 'Business Entity': ['Nordics'],
 'Service Team(s)': ['Tax & Reporting'],
 'Reporter': 'luca.rinaldi@intcom.com',
 'Assignee': 'oliver.varga@intcom.com',
 'Priority': 'low',
 'Urgency': 'low',
 'Impact': 'low',
 'Created date': '2026-03-03 19:36',
 'Status': 'done',
 'Resolution': 'done',
 'Resolution date': '2026-03-16 11:51',
 'All Comments': ['nina.baker@intcom.com: Initial triage assigned to Tax & Reporting and reviewed against the service catalogue.',
  'peter.kuznetsov@intcom.com: We validated the issue against Tax Reporting and checked whether it matched the expected operating conditions.',
  'yasmine.fox@intcom.com:

## Build the DataFrame

List-valued fields (`Affected Business or IT Services`, `Business Entity`, `Service Team(s)`, `All Comments`) are kept as-is; dates are parsed.

In [5]:
df = pd.DataFrame(records)

date_cols = ["Created date", "Resolution date"]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors="coerce")

df["Comment count"] = df["All Comments"].apply(lambda x: len(x) if isinstance(x, list) else 0)

print(df.shape)
df.head()

(20000, 17)


,Work type,Summary,Description,Affected Business or IT Services,Business Entity,Service Team(s),Reporter,Assignee,Priority,Urgency,Impact,Created date,Status,Resolution,Resolution date,All Comments,Comment count
0,Service Request,New license requested for Tax Reporting,A new user requires a license for Tax Reportin...,[Tax Reporting],[Nordics],[Tax & Reporting],luca.rinaldi@intcom.com,oliver.varga@intcom.com,low,low,low,2026-03-03 19:36:00,done,done,2026-03-16 11:51:00,[nina.baker@intcom.com: Initial triage assigne...,3
1,Service Request,New license requested for Outlook & Email,A new user requires a license for Outlook & Em...,[Outlook & Email],[Switzerland],[Enterprise Applications],maia.berg@intcom.com,jack.martin@intcom.com,low,medium,lowest,2026-06-07 07:38:00,done,cannot reproduce,2026-06-09 13:30:00,[louis.belov@intcom.com: Initial triage assign...,3
2,Service Request,Incorrect incident title for service request i...,The submitted title suggests an incident affec...,[Portfolio Accounting],[Switzerland],[Investment Operations],info@extcom_19.com,tania.gupta@intcom.com,lowest,low,medium,2026-06-23 14:00:00,done,clarification,2026-06-28 09:07:00,[marta.marchand@intcom.com: Initial triage ass...,3
3,Service Request,New license requested for Trading Platform,A new user requires a license for Trading Plat...,[Trading Platform],[Germany],[Investment Operations],carlos.ortiz@intcom.com,nora.leclerc@intcom.com,lowest,lowest,low,2026-01-23 06:26:00,done,cannot reproduce,2026-02-05 00:13:00,[ursula.klassen@intcom.com: Initial triage ass...,4
4,Service Request,Access requested for SimCorp Dimension,A user needs access to SimCorp Dimension to co...,[SimCorp Dimension],[Germany],[Enterprise Applications],celine.novak@intcom.com,xena.schmidt@intcom.com,low,lowest,medium,2026-07-08 11:54:00,done,clarification,2026-07-15 22:33:00,[olga.dupont@intcom.com: Initial triage assign...,4


## Quick overview

In [6]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 17 columns):
 #   Column                            Non-Null Count  Dtype         
---  ------                            --------------  -----         
 0   Work type                         20000 non-null  str           
 1   Summary                           20000 non-null  str           
 2   Description                       20000 non-null  str           
 3   Affected Business or IT Services  20000 non-null  object        
 4   Business Entity                   20000 non-null  object        
 5   Service Team(s)                   20000 non-null  object        
 6   Reporter                          20000 non-null  str           
 7   Assignee                          20000 non-null  str           
 8   Priority                          20000 non-null  str           
 9   Urgency                           20000 non-null  str           
 10  Impact                            20000 non-null  str    

In [7]:
# Missing values per column
df.isna().sum().sort_values(ascending=False)

Resolution                          3031
Resolution date                     3031
Work type                              0
Summary                                0
Description                            0
Service Team(s)                        0
Reporter                               0
Affected Business or IT Services       0
Business Entity                        0
Priority                               0
Assignee                               0
Urgency                                0
Impact                                 0
Status                                 0
Created date                           0
All Comments                           0
Comment count                          0
dtype: int64

In [21]:
# Value counts for the key categorical fields
for col in ["Work type", "Priority", "Urgency", "Impact", "Status","Resolution","Summary","Description","Assignee"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False))


Work type:
Work type
Incident           16000
Service Request     4000
Name: count, dtype: int64

Priority:
Priority
lowest     10037
low         6070
medium      1880
high        1629
highest      384
Name: count, dtype: int64

Urgency:
Urgency
lowest     10016
low         5979
medium      2008
high        1618
highest      379
Name: count, dtype: int64

Impact:
Impact
lowest     10064
low         5962
medium      1962
high        1636
highest      376
Name: count, dtype: int64

Status:
Status
done           16969
in progress     2064
open             967
Name: count, dtype: int64

Resolution:
Resolution
cannot reproduce    4322
clarification       4243
done                4236
cancelled           4168
NaN                 3031
Name: count, dtype: int64

Summary:
Summary
External email warning received for Emailed Support Tickets              4212
Email notification received for Emailed Support Tickets                  1211
Automated alert triggered for SimCorp Dimension              

## Assignee analysis

**Question:** is there a pattern in *which* assignee a ticket gets, and can a model predict it?

How we answer it, from cheap to strong:

1. **Distribution:** is the workload even, or do some assignees get far more tickets than others?
2. **Association:** does the assignee depend on any other field (service, team, entity, work type, reporter, dates, comments, resolution)? Measured with the bias-corrected Cramér's V and a chi-square test, with a Bonferroni correction because many fields are tested.
3. **Order and time:** is there round-robin, "same person again" or reporter/assignee structure?
4. **Model:** five models predict the assignee under 5-fold cross-validation. They are compared with the chance level (1 / number of assignees) and with the baseline the pipeline uses today, "most common assignee of the service". A model that beats these has found a pattern. A **permutation test** (shuffled labels) and a **positive control** (predicting the team from the service, which must work) show that the setup itself is sound.

### 1 · Prepare the fields

Every ticket has one value per list field. Numeric and date fields are turned into small categories (hour, weekday, month, resolution-time quintile, comment count) so that any non-linear effect can show up as an association.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import chi2_contingency

SEED = 42
INK, INK_2, MUTED, GRID, AXIS, SURFACE = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7", "#fcfcfb"
ACCENT, ACCENT_2, CONTEXT = "#2a78d6", "#eb6834", "#d9d8d2"
plt.rcParams.update({"axes.facecolor": SURFACE, "figure.facecolor": SURFACE, "axes.edgecolor": AXIS, "axes.labelcolor": INK_2,
                     "xtick.color": INK_2, "ytick.color": INK_2, "text.color": INK, "grid.color": GRID,
                     "axes.spines.top": False, "axes.spines.right": False})


def first(values):
    return values[0] if isinstance(values, list) and values else "(none)"


def commenter(comments, position):
    # comments look like "<email>: <text>"
    return comments[position].split(":")[0] if isinstance(comments, list) and comments else "(none)"


a = pd.DataFrame(index=df.index)
a["Assignee"] = df["Assignee"]
a["Service"] = df["Affected Business or IT Services"].map(first)
a["Team"] = df["Service Team(s)"].map(first)
a["Entity"] = df["Business Entity"].map(first)
for col in ["Work type", "Reporter", "Priority", "Urgency", "Impact", "Status", "Resolution"]:
    a[col] = df[col].fillna("(none)")

a["Created hour"] = df["Created date"].dt.hour.astype("string").fillna("(none)")
a["Created weekday"] = df["Created date"].dt.dayofweek.astype("string").fillna("(none)")
a["Created month"] = df["Created date"].dt.month.astype("string").fillna("(none)")
hours = (df["Resolution date"] - df["Created date"]).dt.total_seconds() / 3600
a["Resolution time"] = pd.qcut(hours, 5, duplicates="drop").astype("string").fillna("(unresolved)")
a["Comment count"] = df["Comment count"].astype(str)
a["First commenter"] = df["All Comments"].map(lambda c: commenter(c, 0))
a["Last commenter"] = df["All Comments"].map(lambda c: commenter(c, -1))
a["Text"] = df["Summary"].fillna("") + " " + df["Description"].fillna("")

FIELDS = [c for c in a.columns if c not in ("Assignee", "Text")]
y = a["Assignee"]
N_ASSIGNEES = y.nunique()
print(f"{len(a):,} tickets, {N_ASSIGNEES} assignees, chance level 1/{N_ASSIGNEES} = {1 / N_ASSIGNEES:.2%}")
print(f"{len(FIELDS)} candidate fields:", ", ".join(FIELDS))

### 2 · Is the workload even?

Under a uniform random choice every assignee gets about `n / #assignees` tickets, give or take the binomial noise. The grey band is the 95 % range of that noise. Bars inside it are indistinguishable from "picked at random with equal probability".

Each team also has all assignees, so the assignee is **not** restricted to the ticket's team (checked in the second output).

In [ ]:
from scipy.stats import binom

counts = y.value_counts()
n, k = len(y), N_ASSIGNEES
low, high = binom.ppf([0.025, 0.975], n, 1 / k)

fig, ax = plt.subplots(figsize=(9, 0.28 * k + 1.2))
pos = np.arange(k)
ax.axvspan(low, high, color=CONTEXT, alpha=0.6, label=f"95 % range if uniform ({low:.0f}–{high:.0f})")
ax.barh(pos, counts.values, height=0.7, color=ACCENT)
ax.set_yticks(pos, [s.split("@")[0] for s in counts.index], fontsize=8)
ax.invert_yaxis()
ax.set_xlim(counts.min() * 0.9, counts.max() * 1.03)
ax.set_xlabel("tickets")
ax.set_title("Tickets per assignee")
ax.legend(loc="lower right", frameon=False)
ax.grid(axis="x")
ax.set_axisbelow(True)
plt.show()

outside = int(((counts < low) | (counts > high)).sum())
print(f"{outside} of {k} assignees lie outside the uniform 95 % range (about {0.05 * k:.1f} expected by chance alone)")
print("Assignees per team:", a.groupby("Team")["Assignee"].nunique().to_dict())

### 3 · Does the assignee depend on any other field?

For each candidate field the chart shows the **bias-corrected Cramér's V** between that field and the assignee: 0 means independent, 1 means the field determines the assignee. Bias correction matters here: with 30 assignees and a field such as *Reporter* with ~90 values, the plain V is above 0 even for pure noise.

Colour marks fields that pass the chi-square test **after Bonferroni correction** (`p < 0.05 / number of fields`). Testing 17 fields at 5 % each would otherwise produce a false hit about every other run.

In [ ]:
def cramers_v(x, target):
    table = pd.crosstab(x, target)
    chi2, p, _, _ = chi2_contingency(table, correction=False)
    total = table.values.sum()
    rows, cols = table.shape
    phi2 = max(0.0, chi2 / total - (cols - 1) * (rows - 1) / (total - 1))
    rows_c, cols_c = rows - (rows - 1) ** 2 / (total - 1), cols - (cols - 1) ** 2 / (total - 1)
    return np.sqrt(phi2 / max(1e-12, min(rows_c - 1, cols_c - 1))), p


assoc = pd.DataFrame([(f, *cramers_v(a[f], y)) for f in FIELDS], columns=["Field", "V", "p"]).sort_values("V", ascending=False)
ALPHA = 0.05 / len(FIELDS)
assoc["significant"] = assoc["p"] < ALPHA

fig, ax = plt.subplots(figsize=(9, 0.4 * len(assoc) + 1.2))
pos = np.arange(len(assoc))
ax.barh(pos, assoc["V"], height=0.65, color=[ACCENT_2 if s else CONTEXT for s in assoc["significant"]])
for yi, (v, p) in enumerate(zip(assoc["V"], assoc["p"])):
    ax.annotate(f"V={v:.3f}  p={p:.2f}", (v, yi), xytext=(6, 0), textcoords="offset points", va="center", color=INK_2, fontsize=9)
ax.set_yticks(pos, assoc["Field"])
ax.invert_yaxis()
ax.set_xlim(0, 1)
ax.set_xlabel("bias-corrected Cramér's V  (1 = field determines the assignee)")
ax.set_title(f"Association with Assignee  (significant if p < {ALPHA:.4f})")
ax.grid(axis="x")
ax.set_axisbelow(True)
plt.show()

print("Significant after Bonferroni:", assoc.loc[assoc["significant"], "Field"].tolist() or "none")

### 4 · Order and time: round-robin, repeats, reporter

A pattern that no single field shows can still live in the *sequence*:

- **Round-robin or "same person again":** how often is the assignee of ticket *i* the same as that of ticket *i − lag* (tickets sorted by creation time)? If assignees are drawn independently, the rate equals `Σ pᵢ²`, the chance of two independent draws matching. The band is the 95 % noise range.
- **Reporter = assignee:** compared with the rate expected if reporter and assignee were independent.

In [ ]:
ordered = y.loc[df["Created date"].sort_values(kind="stable").index].to_numpy()
p_match = float((y.value_counts(normalize=True) ** 2).sum())

lags = np.arange(1, 31)
rates = np.array([(ordered[lag:] == ordered[:-lag]).mean() for lag in lags])
band = 1.96 * np.sqrt(p_match * (1 - p_match) / len(ordered))

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.axhspan(p_match - band, p_match + band, color=CONTEXT, alpha=0.6, label="95 % range if independent")
ax.plot(lags, rates, color=ACCENT, lw=2, marker="o", ms=5, mec=SURFACE, mew=1.2, label="observed")
ax.set_xlabel("lag (tickets, sorted by creation time)")
ax.set_ylabel("share with the same assignee")
ax.set_title("Same assignee again after `lag` tickets")
ax.legend(frameon=False)
ax.grid(axis="y")
ax.set_axisbelow(True)
plt.show()
print(f"lags outside the band: {int(((rates < p_match - band) | (rates > p_match + band)).sum())} of {len(lags)} (about {0.05 * len(lags):.1f} expected by chance)")

reporter_share = a["Reporter"].value_counts(normalize=True)
expected = float((y.value_counts(normalize=True) * reporter_share.reindex(y.value_counts().index).fillna(0)).sum())
print(f"assignee == reporter: {(a['Assignee'] == a['Reporter']).mean():.2%}   expected if independent: {expected:.2%}")

### 5 · Model

The model has to answer: *given everything we know about a ticket at intake, who is the assignee?*

**Setup**

| Item | Choice |
|---|---|
| Target | `Assignee` (30 classes) |
| Validation | 5-fold stratified cross-validation. Every score is on tickets the model has not seen. |
| Metrics | **top-1 accuracy** (the suggestion is right) and **top-3 accuracy** (the right person is among the three most likely) |
| Chance | top-1 = 1 / 30 ≈ 3.3 %, top-3 = 3 / 30 = 10 % |

**Models, from simple to rich**

1. **Most common overall:** always the assignee with the most tickets. The floor.
2. **Most common per service:** what the pipeline does today (`RoutingStatistics`): the historical majority assignee of the ticket's service (team).
3. **Logistic regression on the routing fields:** service, team, entity, work type, reporter.
4. **Logistic regression on the text:** TF-IDF (1–2-grams) over summary + description.
5. **Logistic regression on everything:** all fields above plus dates, comments, resolution and text.

Fields such as resolution, status and the comments only exist after the work is done. They are included on purpose: if they were a *cause* of the assignee they would show up here. A model that uses them could not be used at intake, so a useful signal in them would still need a different question.

Regularisation (`C`) is small so that the models cannot memorise noise.

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder


class GroupFrequency(BaseEstimator, ClassifierMixin):
    """Predicts the target frequencies of the ticket's group. `column=None` means one group (overall majority)."""

    def __init__(self, column=None):
        self.column = column

    def _keys(self, X):
        return X[self.column] if self.column else pd.Series("all", index=X.index)

    def fit(self, X, y):
        self.classes_ = np.array(sorted(pd.unique(y)))
        table = pd.crosstab(self._keys(X), pd.Series(np.asarray(y), index=X.index)).reindex(columns=self.classes_, fill_value=0)
        self.freq_ = table.div(table.sum(axis=1), axis=0)
        self.prior_ = (table.sum() / table.values.sum()).to_numpy()
        return self

    def predict_proba(self, X):
        found = self.freq_.reindex(self._keys(X))
        return found.fillna(pd.Series(self.prior_, index=self.classes_)).to_numpy()

    def predict(self, X):
        return self.classes_[self.predict_proba(X).argmax(axis=1)]


def one_hot(columns):
    return ColumnTransformer([("c", OneHotEncoder(handle_unknown="ignore", min_frequency=5), columns)])


ROUTING = ["Service", "Team", "Entity", "Work type", "Reporter"]
text_only = ColumnTransformer([("t", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True), "Text")])
everything = ColumnTransformer([
    ("c", OneHotEncoder(handle_unknown="ignore", min_frequency=5), FIELDS),
    ("t", TfidfVectorizer(ngram_range=(1, 2), min_df=3, sublinear_tf=True), "Text"),
])

MODELS = {
    "1 Most common overall": GroupFrequency(),
    "2 Most common per service\n(pipeline today)": GroupFrequency("Service"),
    "3 Logistic regression\nrouting fields": make_pipeline(one_hot(ROUTING), LogisticRegression(C=0.05, max_iter=300)),
    "4 Logistic regression\ntext": make_pipeline(text_only, LogisticRegression(C=0.3, max_iter=300)),
    "5 Logistic regression\nall fields + text": make_pipeline(everything, LogisticRegression(C=0.05, max_iter=300)),
}
cv = StratifiedKFold(5, shuffle=True, random_state=SEED)


def top_k(proba, classes, truth, k):
    best = np.argsort(-proba, axis=1)[:, :k]
    return float(np.mean([t in set(classes[row]) for t, row in zip(truth, best)]))


rows = []
for name, model in MODELS.items():
    proba = cross_val_predict(model, a, y, cv=cv, method="predict_proba", n_jobs=-1)
    classes = np.array(sorted(y.unique()))
    rows.append({"Model": name, "top-1": top_k(proba, classes, y.to_numpy(), 1), "top-3": top_k(proba, classes, y.to_numpy(), 3)})
scores = pd.DataFrame(rows).set_index("Model")
scores.style.format("{:.2%}")

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.2))
x = np.arange(len(scores))
w = 0.38
ax.bar(x - w / 2, scores["top-1"], w, color=ACCENT, label="top-1")
ax.bar(x + w / 2, scores["top-3"], w, color=ACCENT_2, label="top-3")
ax.axhline(1 / N_ASSIGNEES, color=ACCENT, ls="--", lw=1, label=f"chance top-1 ({1 / N_ASSIGNEES:.1%})")
ax.axhline(3 / N_ASSIGNEES, color=ACCENT_2, ls="--", lw=1, label="chance top-3 (10 %)")
for xi, (t1, t3) in enumerate(zip(scores["top-1"], scores["top-3"])):
    ax.annotate(f"{t1:.1%}", (xi - w / 2, t1), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8.5, color=INK_2)
    ax.annotate(f"{t3:.1%}", (xi + w / 2, t3), xytext=(0, 3), textcoords="offset points", ha="center", fontsize=8.5, color=INK_2)
ax.set_xticks(x, scores.index, fontsize=8)
ax.set_ylim(0, 0.16)
ax.yaxis.set_major_formatter(lambda v, _: f"{v:.0%}")
ax.set_title("Predicting the assignee (5-fold cross-validation)")
ax.legend(frameon=False, ncol=2, fontsize=8.5)
ax.grid(axis="y")
ax.set_axisbelow(True)
plt.show()

### 6 · Can the result be trusted?

Two checks make sure that "no pattern" is a finding and not a broken setup:

- **Permutation test:** shuffle the assignees (this destroys any real link to the ticket, but keeps the class sizes) and score the same model again. If the real score is not clearly above the shuffled scores, the model has learned nothing beyond chance. Uses model 5, the richest one.
- **Positive control:** use the *same* pipeline to predict a target that **is** determined by another field, the team from the service (1:1 in this export). It must score close to 100 %. If it does, the setup can find a pattern when there is one.

In [ ]:
N_PERM = 10
model5 = MODELS["5 Logistic regression\nall fields + text"]
real = cross_val_score(model5, a, y, cv=cv, n_jobs=-1).mean()

rng = np.random.default_rng(SEED)
null = np.array([
    cross_val_score(model5, a, pd.Series(rng.permutation(y.to_numpy()), index=a.index), cv=cv, n_jobs=-1).mean()
    for _ in range(N_PERM)
])
p_value = (1 + (null >= real).sum()) / (1 + N_PERM)
print(f"real accuracy {real:.2%} | shuffled labels {null.mean():.2%} ± {null.std():.2%} (n={N_PERM}) | permutation p = {p_value:.2f}")

control_model = make_pipeline(one_hot(["Service"]), LogisticRegression(C=1.0, max_iter=300))
control = cross_val_score(control_model, a, a["Team"], cv=cv, n_jobs=-1).mean()
print(f"positive control, predict Team from Service: {control:.1%} accuracy")

### 7 · Conclusion: how the assignee is chosen

Read the outputs above together:

- **No pattern found.** The assignee is statistically independent of every field in the export: service, team, entity, work type, reporter, priority / urgency / impact, status, resolution, creation hour / weekday / month, resolution time, comments and the ticket text. No field passes the Bonferroni-corrected test, and the bias-corrected Cramér's V is 0.023 or lower for all of them. The closest is *Business Entity* (V ≈ 0.02, raw p ≈ 0.006). That is too weak to matter: the highest share of any assignee inside an entity is about 4 % against 3.3 % overall, and it does not survive the correction for 17 tests.
- **No sequence structure either:** no round-robin and no "same person again" effect, and the assignee is not the reporter more often than expected.
- **Workload is even:** each assignee gets about 1/30 of the tickets, and every assignee appears in every team, so the assignee is not tied to the ticket's team.
- **The models confirm it.** The model on all fields scores at chance level. It is indistinguishable from its own shuffled-label baseline, while the positive control (team from service) reaches ~100 %. So the setup can find a pattern, and here there is none.
- **Best possible strategy:** "most common overall" and "most common per service" are equally good. All rules land at ~3.5–3.6 % top-1 and ~10–11 % top-3, i.e. chance.

**What this means for the triage pipeline (decided 2026-09-25):** the assignee in this data is effectively random (or set by information that is not in the export, e.g. shift plans or availability). No LLM or heavier model is used for it. The pipeline now suggests the person with the **fewest tickets assigned** (training tickets plus stored suggestions, every new suggestion counts immediately). That spreads the work evenly and costs no accuracy, since every rule is at chance. Expect assignee accuracy near 1/30 and show the suggestion as low-confidence to reviewers. A real assignment rule would need data that is not in this export, for example who is on duty, current workload or skills.
